In [ ]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset

# Fetch

In [16]:
import kagglehub

In [17]:
temp_dir = Path.cwd() / ".temp"

if temp_dir.parent.name != "model":
    raise RuntimeError()

temp_dir.mkdir(exist_ok=True)
temp_dir

PosixPath('/Users/cube/source/dhbw/ml-digits/model/.temp')

In [18]:
dataset_path = (
    Path(
        kagglehub.dataset_download(
            "metricasecuador/handwritten-digits-version-1-hwd-v1",
            output_dir=str(temp_dir / "dataset"),
        )
    )
    / "HWD-V1"
)
dataset_path

PosixPath('/Users/cube/source/dhbw/ml-digits/model/.temp/dataset/HWD-V1')

# PyTorch

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


# Dataset

In [21]:
RESOLUTION = 32

In [22]:
def dataset_all_split(
    dataset: dict[int, list[Path]], split: float, random_state: int | None = None
):
    """
    Return two split dicts from DATASET_ALL
    """
    train, test = {}, {}

    rng = np.random.default_rng(random_state)

    for digit, items in dataset.items():
        shuffled_items = list(items)
        rng.shuffle(shuffled_items)

        split_idx = int(len(shuffled_items) * split)
        train[digit] = shuffled_items[:split_idx]
        test[digit] = shuffled_items[split_idx:]

    return train, test


def dataset_dict_to_list(dataset_dict):
    return [(item, digit) for digit, items in dataset_dict.items() for item in items]

In [23]:
DATASET_ITEMS_ALL = {
    digit: list(dataset_path.glob(f"HWD-V1-Standard/{digit}/*.png"))
    for digit in range(10)
}

DATASET_ITEMS_TRAIN, DATASET_ITEMS_TEST = dataset_all_split(DATASET_ITEMS_ALL, 0.7, random_state=99)
DATASET_ITEMS_ALL_LIST, DATASET_ITEMS_TRAIN_LIST, DATASET_ITEMS_TEST_LIST = (
    dataset_dict_to_list(DATASET_ITEMS_ALL),
    dataset_dict_to_list(DATASET_ITEMS_TRAIN),
    dataset_dict_to_list(DATASET_ITEMS_TEST),
)

len(DATASET_ITEMS_TRAIN_LIST), len(DATASET_ITEMS_TEST_LIST)

(105000, 45000)

In [24]:
class DigitDataset(Dataset):
    def __init__(self, *, items: list[tuple[Path, int]], r: int):
        self.r = r
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx: int):
        item_path, label = self.items[idx]

        img = Image.open(item_path).convert("L")
        img = ImageOps.invert(img)

        img = img.resize((self.r, self.r))
        img.save(temp_dir / "test_img.png")

        data = np.array(img.get_flattened_data(), dtype=np.float32) / 255
        tensor = torch.tensor(data, dtype=torch.float32)
        tensor = tensor.view(1, self.r, self.r)

        return tensor, label


ds = DigitDataset(items=DATASET_ITEMS_ALL_LIST, r=RESOLUTION)
ds.__getitem__(99999)

(tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]]),
 6)

# Model

In [ ]:
@dataclass(kw_only=True)
class TrainResult:
    dataset: Dataset
    model: nn.Module
    epoch: int
    loss: float
    accuracy: float

In [ ]:
def train(
    *, dataset: DigitDataset, model: nn.Module, epochs: int, echo=True
) -> TrainResult:
    loader = DataLoader[DigitDataset](dataset, batch_size=32, shuffle=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        total_loss = 0
        correct = 0

        for X_batch, Y_batch in loader:
            Y_pred: torch.Tensor = model(X_batch)
            loss = loss_fn(Y_pred, Y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * X_batch.size(0)
            correct += (Y_pred.argmax(1) == Y_batch).sum().item()

        n = len(dataset)
        calc_loss = total_loss / n
        calc_accuracy = correct / n

        if echo:
            print(
                f"Epoch {epoch + 1:2d}  loss={total_loss / n:.4f}  acc={correct / n:.2%}"
            )

    return TrainResult(
        dataset=dataset,
        model=model,
        epoch=epoch,
        loss=calc_loss,
        accuracy=calc_accuracy,
    )

In [25]:
class VerySimpleLinearModel(nn.Module):
    def __init__(self, *, r: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),  # (1, R, R) -> RxR
            nn.Linear(r * r, 10),  # RxR -> 10 digit scores
            nn.Softmax(),
        )

    def forward(self, x):
        return self.net(x)

In [27]:
dataset_train = DigitDataset(items=DATASET_ITEMS_TRAIN_LIST, r=RESOLUTION)
loader = DataLoader[DigitDataset](dataset_train, batch_size=32, shuffle=True)

result = train(dataset=dataset_train, model=VerySimpleLinearModel(r=RESOLUTION), epochs=30)

result


/Users/cube/source/dhbw/ml-digits/.venv/lib/python3.14/site-packages/torch/nn/modules/module.py:1779: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


Epoch  1  loss=1.5388  acc=95.29%
Epoch  2  loss=1.4932  acc=97.49%
Epoch  3  loss=1.4876  acc=97.86%
Epoch  4  loss=1.4848  acc=98.06%
Epoch  5  loss=1.4831  acc=98.18%
Epoch  6  loss=1.4817  acc=98.28%
Epoch  7  loss=1.4806  acc=98.36%
Epoch  8  loss=1.4797  acc=98.45%
Epoch  9  loss=1.4790  acc=98.49%
Epoch 10  loss=1.4784  acc=98.54%
Epoch 11  loss=1.4780  acc=98.57%
Epoch 12  loss=1.4775  acc=98.62%
Epoch 13  loss=1.4771  acc=98.63%
Epoch 14  loss=1.4767  acc=98.67%
Epoch 15  loss=1.4763  acc=98.70%
Epoch 16  loss=1.4761  acc=98.71%
Epoch 17  loss=1.4758  acc=98.74%
Epoch 18  loss=1.4756  acc=98.76%
Epoch 19  loss=1.4753  acc=98.78%
Epoch 20  loss=1.4751  acc=98.80%
Epoch 21  loss=1.4749  acc=98.82%
Epoch 22  loss=1.4746  acc=98.84%
Epoch 23  loss=1.4745  acc=98.84%
Epoch 24  loss=1.4743  acc=98.86%
Epoch 25  loss=1.4742  acc=98.88%
Epoch 26  loss=1.4740  acc=98.90%
Epoch 27  loss=1.4738  acc=98.90%
Epoch 28  loss=1.4737  acc=98.91%
Epoch 29  loss=1.4735  acc=98.94%
Epoch 30  loss

TrainResult(dataset=<__main__.DigitDataset object at 0x1156d3b10>, model=VerySimpleLinearModel(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1024, out_features=10, bias=True)
    (2): Softmax(dim=None)
  )
), epoch=29, loss=1.473420037042527, accuracy=0.9893142857142857)